In [8]:
import pandas as pd
from snownlp import SnowNLP
import json

## Agent 2: Classification task

该任务既可以使用LLM直接输出结果，也可以通过batch方式；batch方式可以节约token成本；SUMM-Agent采用Batch推理。

In [9]:
model_name = 'qwen-plus-latest'
sen_names = pd.read_csv('llm_sensitive_name.csv')['name'].tolist()

In [20]:
categories = [
    {'value': '政治与政策', 'description': '任免、廉政反腐、纪律建设、政府工作报告、党建、理论学习、八项规定、公务员培训、国家政策等。'},
    {'value': '经济与金融', 'description': '刺激消费、产业升级、企业服务、经济发展、经济规划、高质量发展等。'},
    {'value': '文化与旅游', 'description': '传统节日、剧团、美食、历史文学作品、历史建筑、非遗、文化活动、旅游产业、演唱会等。'},
    {'value': '体育赛事', 'description': '体育赛事。'},
    {'value': '教育与人才', 'description': '创业、吸引人才、大中小学及幼儿园教育、荣誉表彰、历史人物榜样、主题教育等。'},
    {'value': '医疗与公共健康', 'description': '医疗保险、流感、病毒、医院、中医药、保健、食品安全等。'},
    {'value': '环境与可持续发展', 'description': '动植物保护、植树造林、湿地恢复、生态治理、城市环境治理等。'},
    {'value': '交通与基础设施', 'description': '交通工程、物流运输、货运服务、交通管制、市政工程等。'},
    {'value': '法律与治安管理', 'description': '市场监管、网络安全、反诈宣传、社会治安、治安警局宣传等。'},
    {'value': '应急与安全', 'description': '含天气预警、极端天气、防火提醒、消防安全、海事应急等。'},
    {'value': '科技与创新', 'description': '科技创新等。'},
    {'value': '就业与培训', 'description': '工作机会、就业培训、职业教育、求职等。'},
    {'value': '民生与便民服务', 'description': '社会工作与便民服务、民生福利、自助服务、电子政务等。'},
    {'value': '其他', 'description': '不属于以上任何类别的内容。'},
]

def construct_prompt(title):
    role = '你是一名专业的新闻标注员，擅长新闻分类任务。'
    prompt = '请将新闻分配到以下14个类别中，每条新闻只可以属于多个类别。\n'
    prompt += '**新闻类别**：'
    for category in categories:
        prompt += f'{category["value"]}：{category["description"]}\n'

    prompt += '每条新闻包含新闻标题。\n'
    prompt += '**任务要求**：\n'
    prompt += '1. 根据新闻的内容，确定最相关的类别。\n'
    prompt += '2. 每条新闻至少分配 1 个类别\n'
    prompt += '3. 不可以修改任何分类的名称，必须和提供的分类名称保持100%一致，也不可以增加分类，只能从指定的新闻类别中挑选最合适的类别。\n'
    prompt += '4. 只能输出结果，不允许提供任何解释和推理过程的回答，示例：政治与政策\n'

    
    prompt += '以下是新闻标题：\n'
    prompt += f'{title}。\n'
    prompt += '请根据以上内容输出对应的新闻分类标签。'

    messages = [
        {'role': 'system', 'content': role},
        {'role': 'user', 'content': prompt.strip()}
    ]
    
    return messages

In [21]:
def extract_info(file_path):
    ds = []  
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                item = json.loads(line.strip())
                _id = item.get('custom_id') 
                text = item.get('response').get('body').get('choices')[0].get('message').get('content')
                if text:
                    text = text.strip()
                    ds.append([_id, text])
            except json.JSONDecodeError:
                print(f'无法解析文件 {file_path} 中的一行: {line.strip()}')

    temp = pd.DataFrame(ds, columns=['idx', 'title'])
    return temp

In [24]:
def builder_task(df, filename):
    with open('./data/llm-gen-class/' + filename + model_name + '.jsonl', 'w', encoding='utf-8') as fout:
        for i, row in df.iterrows():
            title = row['title']
            for name in sen_names:
                title = title.replace(name, '领导人')
                
            llm_bu = construct_prompt(title)
            body = {'model': model_name, 'messages': llm_bu, 'temperature': 0.1}
            request = {'custom_id': row['idx'], 'method': 'POST', 'url': '/v1/chat/completions', 'body': body}
            fout.write(json.dumps(request, separators=(',', ':'), ensure_ascii=False) + '\n')



In [25]:
# workflow: agent 1 task1 -> agent 2 task1 
local_success = extract_info('./data/llm-gen-summ-results/local_success.jsonl')
builder_task(local_success, 'local_media')

In [26]:
non_local_success = extract_info('./data/llm-gen-summ-results/non_success.jsonl')
builder_task(non_local_success, 'other_media')
len(non_local_success)

4304